# Fine-Tuning Arsitektur Visi Modern: ConvNeXt pada Dataset Bunga

**Buku:** *Deep Learning dengan Python* — Bab 10, Vision Transformer dan Arsitektur Visi Modern

ConvNeXt adalah convnet yang "dimodernkan" dengan resep desain dari ViT. Kita memakai bobot pralatih ImageNet dengan strategi dua tahap: latih head klasifikasi dulu, lalu *fine-tune* seluruh jaringan dengan *learning rate* kecil.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Notebook ini memakai backend **TensorFlow** (dibutuhkan oleh `tf.data`/`tf.GradientTape`/ekspor).

**Perkiraan waktu (GPU T4):** ±15 menit

**Isi notebook:**
- Dataset `flower_photos` (5 kelas, ±3.700 gambar) dengan pembagian latih/validasi/uji.
- Tahap 1: basis dibekukan, hanya head yang dilatih.
- Tahap 2: seluruh jaringan dibuka dengan *learning rate* 1e-5.
- Matriks konfusi pada data uji.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf

## 1. Data

In [ ]:
import tarfile, pathlib, numpy as np
from keras import layers

jalur = keras.utils.get_file("flower_photos.tgz",
    origin="https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz")
if not pathlib.Path("data/flower_photos").exists():
    with tarfile.open(jalur) as berkas_tar:
        berkas_tar.extractall("data")

UKURAN = (224, 224)
umum = dict(validation_split=0.3, seed=42, image_size=UKURAN, batch_size=32)
ds_latih = keras.utils.image_dataset_from_directory("data/flower_photos", subset="training", **umum)
ds_sisa = keras.utils.image_dataset_from_directory("data/flower_photos", subset="validation", **umum)
kelas = ds_latih.class_names
n = int(ds_sisa.cardinality())
ds_val, ds_uji = ds_sisa.take(n // 2), ds_sisa.skip(n // 2)
ds_latih, ds_val, ds_uji = (d.prefetch(tf.data.AUTOTUNE) for d in (ds_latih, ds_val, ds_uji))
print(kelas)

## 2. Model: ConvNeXt-Tiny pralatih + head baru
ConvNeXt di Keras sudah menyertakan prapemrosesan, sehingga masukan cukup piksel 0–255.

In [ ]:
basis = keras.applications.ConvNeXtTiny(include_top=False, weights="imagenet",
                                        input_shape=UKURAN + (3,), pooling="avg")
basis.trainable = False

augmentasi = keras.Sequential([layers.RandomFlip("horizontal"),
                               layers.RandomRotation(0.1),
                               layers.RandomZoom(0.15)], name="augmentasi")

masukan = keras.Input(shape=UKURAN + (3,))
x = augmentasi(masukan)
x = basis(x, training=False)        # BN/LayerNorm tetap mode inferensi, bahkan saat basis dibuka
x = layers.Dropout(0.3)(x)
model = keras.Model(masukan, layers.Dense(len(kelas), activation="softmax")(x))
model.summary()

## 3. Tahap 1: latih head saja

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(ds_latih, validation_data=ds_val, epochs=5)

## 4. Tahap 2: *fine-tune* seluruh jaringan
Setelah mengubah `trainable`, model **wajib di-*compile* ulang**.

In [ ]:
basis.trainable = True
model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(ds_latih, validation_data=ds_val, epochs=5,
          callbacks=[keras.callbacks.ModelCheckpoint("convnext_bunga.keras", save_best_only=True)])

## 5. Evaluasi pada data uji

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

model = keras.models.load_model("convnext_bunga.keras")
y_benar, y_pred = [], []
for gambar, label in ds_uji:
    y_benar.append(label.numpy())
    y_pred.append(model.predict(gambar, verbose=0).argmax(axis=1))
y_benar, y_pred = np.concatenate(y_benar), np.concatenate(y_pred)

print(classification_report(y_benar, y_pred, target_names=kelas, digits=3))
ConfusionMatrixDisplay.from_predictions(y_benar, y_pred, display_labels=kelas, xticks_rotation=45)
plt.show()

**Eksperimen:** ganti `ConvNeXtTiny` dengan `EfficientNetV2B0` atau `ResNet50` (perhatikan fungsi `preprocess_input` masing-masing), lalu bandingkan akurasi dan waktu per epoch.